# Instructions

Make a copy of this colab. Answer all questions by writing code. Run all cells and save output. Download .ipynb and submit it in canvas.

Please check back often for any updates.

2026/04/08 3pm: Initial version.

# Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms.v2 as T
from torch.utils.data import TensorDataset, DataLoader

# CIFAR-10 Data

In [6]:
# https://en.wikipedia.org/wiki/CIFAR-10
# https://www.cs.toronto.edu/~kriz/cifar.html
# https://docs.pytorch.org/vision/main/generated/torchvision.datasets.CIFAR10.html


classes = ['plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

toTensor = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
])

trainset = torchvision.datasets.CIFAR10(root='datasets', train=True,
                                        download=True, transform=toTensor)

testset = torchvision.datasets.CIFAR10(root='datasets', train=False,
                                       download=True, transform=toTensor)


In [7]:
# Q1 (1 points) What are the shapes of train and test?
print(f"Train shape: {trainset.data.shape}")
print(f"Test shape: {testset.data.shape}")

Train shape: (50000, 32, 32, 3)
Test shape: (10000, 32, 32, 3)


In [4]:
# Q2 (optional) Display first 40 images in [4, 10] grid with labels

# Add code using plt.imshow().


In [8]:
from IPython.testing import test
# Q3 (1 point) Create a dataloader for train and test.

train_dataset = TensorDataset(torch.stack([x[0] for x in trainset], dim=0),
                              torch.stack([torch.tensor(x[1]) for x in trainset], dim=0))

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataset = TensorDataset(torch.stack([x[0] for x in testset], dim=0), torch.stack([torch.tensor(x[1]) for x in testset], dim=0))
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)



# Build a multiclass classifier

In [13]:
in_features = 32 * 32 * 3
out_features = len(classes)

# Q4.a - Model, Loss, and Optimizer
torch.manual_seed(0)
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(in_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, out_features)
)

learning_rate = 0.02
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
loss_fn = nn.CrossEntropyLoss()

# Q4.b - Training Loop for 3 Epochs with Accuracy
epochs = 3

for epoch in range(epochs):
    running_loss = 0.0
    correct = 0
    total = 0


    model.train()
    for images, labels in train_dataloader:
        # Forward pass
        outputs = model(images)
        loss = loss_fn(outputs, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Calculate accuracy metrics
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        running_loss += loss.item()

    # Print accuracy at each epoch
    epoch_accuracy = 100 * correct / total
    print(f"Epoch {epoch + 1}/{epochs} - Loss: {running_loss/len(train_dataloader):.4f}, Accuracy: {epoch_accuracy:.2f}%")




Epoch 1/3 - Loss: 1.9977, Accuracy: 27.04%
Epoch 2/3 - Loss: 1.7866, Accuracy: 36.11%
Epoch 3/3 - Loss: 1.7016, Accuracy: 38.96%


In [15]:
!pip install -q torchmetrics
import torchmetrics

In [18]:
# accuracy = torchmetrics.... Add code

accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=len(classes))

model.eval()
with torch.no_grad():
  for images, labels in test_dataloader:
    outputs = model(images)
    _, predicted = torch.max(outputs.data, 1)
    accuracy.update(predicted, labels)

print(f"Test Accuracy: {accuracy.compute() * 100:.2f}%")

Test Accuracy: 41.11%


In [21]:
for epoch in range(3):
  running_loss = 0.0
  correct = 0
  total = 0
  accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=len(classes))

  model.train()

  for images, labels in train_dataloader:
    outputs = model(images)
    loss = loss_fn(outputs, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()
    running_loss += loss.item()
  epoch_accuracy = 100 * correct / total
  print(f"Epoch {epoch + 1}/{epochs} - Loss: {running_loss/len(train_dataloader):.4f}, Accuracy: {epoch_accuracy:.2f}%")

Epoch 1/3 - Loss: 1.5397, Accuracy: 45.13%
Epoch 2/3 - Loss: 1.5164, Accuracy: 45.91%
Epoch 3/3 - Loss: 1.4962, Accuracy: 46.46%


In [22]:
# Q5 (1 point) How many trainable params are there in your model?

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters: {trainable_params}")

Number of trainable parameters: 156100
